In [227]:
import numpy as np 

In [228]:
tags = ["Noun", "Verb" , "adverb"] 

In [229]:
words = ["Vidhya", "is", "reading", "silently"]

# Convert words into indices 
word_to_index = { 
"Vidhya": 0, 
"is" : 1,
"reading": 2,
"silently" : 3
} 

In [230]:
init_prob = np.array([ 
    0.5, # Noun 
    0.3, # Verb 
    0.2 # adverb
]) 

In [231]:
transition = np.array([
    [0.2, 0.7, 0.1],   # Noun -> Verb is common
    [0.1, 0.8, 0.1],   # Verb -> Verb (is -> reading)
    [0.1, 0.2, 0.7]    # Object -> Object
])

In [232]:
emission = np.array([
    [0.8, 0.1, 0.05, 0.05],   # Noun
    [0.05, 0.6, 0.3, 0.05],   # Verb
    [0.05, 0.05, 0.05, 0.85]  # Object
])

In [233]:
log_init = np.log(init_prob + 1e-10) 
log_trans = np.log(transition + 1e-10) 
log_emit = np.log(emission + 1e-10) 

In [234]:
def viterbi(sentence): 
    # Number of words 
    T = len(sentence) 
    # Number of tags 
    N = len(tags) 
    # DP table 
    dp = np.full((N, T), -np.inf) 
    # Backpointer table 
    backpointer = np.zeros((N, T), dtype=int) 

    # Calculate probability for the first word
    dp[:, 0] = log_init + log_emit[:, sentence[0]] 
    
    # ---------------- Recursion ---------------- # Process remaining words 
    for t in range(1, T): 
        # Check every current tag 
        for j in range(N): 
            # Calculate score from every previous tag 
            scores = ( dp[:, t-1] 
            + log_trans[:,j ] 
            + log_emit[j, sentence[t]] 
            ) 
            # Store best previous tag 
            backpointer[j, t] = np.argmax(scores) 
            # Store best score 
            dp[j, t] = np.max(scores) 

    # Best tag for last word 
    best_path = [np.argmax(dp[:, -1])] 
    # Trace backwards 
    for t in range(T-1, 0, -1): 
        best_path.append(backpointer[best_path[-1], t]) 
    # Reverse to get correct order 
    best_path.reverse() 
    return best_path

In [235]:
sentence = ["Vidhya", "is", "reading", "silently"] 
# Convert words into indices 
sentence_index = [word_to_index[word] for word in sentence] 
# Run Viterbi 
best_tags = viterbi(sentence_index) 

In [236]:
print("Sentence:", sentence) 
print("\nPredicted Tags:") 
for word, tag in zip(sentence, best_tags): 
    print(word, "->", tags[tag]) 

Sentence: ['Vidhya', 'is', 'reading', 'silently']

Predicted Tags:
Vidhya -> Noun
is -> Verb
reading -> Verb
silently -> adverb
